# Surrogate Model Convergence
This notebook provides a template for checking and plotting the convergence of surrogate models based on the number of points sampled randomly.
Multiple random samples are taken from each function to check method accuracy relative to number of samples.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import multi_fidelity_functions as mfs
from SurrogateModels import kriging

### Convergence Over Different Dimensions
Test a set of functions from 1-8 dimensions for accuracy in fitting a function

In [2]:
# Set some storage ontainers
error = {}
points = {}

In [70]:
import time


def create_test_points(func: mfs.MultiFidelityTemplate, n=2000):
    # Create a set of samples to create a ground truth
    X_test = func.sample_random(n)
    y_test = func.high_fidelity(X_test)
    return X_test, y_test


def generate_kriging_model(
    name,
    func: mfs.MultiFidelityTemplate,
    n,
    seed,
    X_test,
    y_test,
    x_transform=True,
    bounds=None,
):
    X_train = func.sample_random(n, seed=seed)
    # X_train = func.sample_uniform(n)
    y_train = func.high_fidelity(X_train)
    # Setup timer to gauge model training time
    t0 = time.perf_counter()
    model = kriging.Kriging(
        X_train,
        y_train,
        nugget=1e-7,
        scale_input=x_transform,
        bounds=bounds,
    )
    model_training_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    error = model.model_error(X_test, y_test)
    evaluation_time = time.perf_counter() - t0
    model_error = model.model_error(X_test, y_test)
    print(
        f"Run {name} :: seed {seed} :: points {n} :: error {error} :: train_time {model_training_time} complete"
    )
    return model, error, model_training_time, evaluation_time


def generate_model_comparisons(
    name,
    func: mfs.MultiFidelityTemplate,
    X_test,
    y_test,
    sample_low=5,
    sample_high=105,
    step_size=5,
    seeds=5,
    model_error={},
    points={},
):
    # Generate a Kriging model of the data and test accuracy using a number of random sample points
    n_points = list(range(sample_low, sample_high, step_size))
    points[name] = []
    model_error[name] = []
    for seed in range(0, seeds):
        for n in n_points:
            X_train = func.sample_random(n, seed=seed)
            # X_train = func.sample_uniform(n)
            y_train = func.high_fidelity(X_train)
            # Setup timer to gauge model training time
            t0 = time.perf_counter()
            model = kriging.Kriging(X_train, y_train, nugget=1e-7)
            model_training_time = time.perf_counter() - t0

            t0 = time.perf_counter()
            error = model.model_error(X_test, y_test)
            evaluation_time = time.perf_counter() - t0
            model_error[name].append(model.model_error(X_test, y_test))
            points[name].append(n)
            print(
                f"Run {name} :: seed {seed} :: points {n} :: error {error} :: train_time {model_training_time} complete"
            )


def plot_error_measure(
    points, error, name, color="blue", marker="o", label="Test", ax=None
):
    if ax == None:
        fig = plt.figure(name)
        ax = fig.add_subplot()
    ax.set_title(name)
    ax.scatter(
        points,
        error,
        color=color,
        label=label,
        marker=marker,
    )
    ax.set_xlabel("Samples")
    ax.set_ylabel("RMS Error")
    ax.legend()
    ax.grid()
    ax.set_yscale("log")
    return ax

In [ ]:
key = "2D"
my_function = mfs.increasing_dimensions[key]()
X_test, y_test = create_test_points(my_function, n=2000)
x_transform = True
y_transform = True
test_bounds = np.array(my_function.bounds, dtype=float)
print(test_bounds)

model, _, _, _ = generate_kriging_model("n=10", my_function, 200, 6, X_test, y_test,x_transform,bounds=my_function.bounds)
ax = model.plot_model(my_function.bounds,"n=10")
ax = model.plot_data(name = "n=10",ax=ax)

[[-3.  3.]
 [-3.  3.]]


TypeError: generate_kriging_model() got multiple values for argument 'bounds'

In [ ]:
# CHECK IF KRIGING MODEL NEEDS OPTIMIZER TUNING ON 1D CASE
key = "1D"
my_function = mfs.increasing_dimensions[key]()
X_test, y_test = create_test_points(my_function, n=2000)
print(y_test)
generate_model_comparisons(
    key,
    my_function,
    X_test=X_test,
    y_test=y_test,
    sample_low=10,
    sample_high=50,
    step_size=20,
    seeds=1,
    model_error=error,
    points=points,
)

[[ 30.17308688]
 [ 19.14324597]
 [  0.42939817]
 ...
 [103.65291156]
 [ 41.22400597]
 [ 19.57734419]]
Run 8D :: seed 0 :: points 100 :: error 0.7069796098084338 :: train_time 673.1972696000012 complete
Run 8D :: seed 0 :: points 120 :: error 0.7598042732159709 :: train_time 502.4057570999721 complete
Run 8D :: seed 0 :: points 140 :: error 0.21834937207787672 :: train_time 291.26316470000893 complete
Run 8D :: seed 0 :: points 160 :: error 0.2027973966253753 :: train_time 1035.6301258000312 complete
Run 8D :: seed 0 :: points 180 :: error 0.16823405588374624 :: train_time 789.519465899968 complete
Run 8D :: seed 0 :: points 200 :: error 0.09788645421046353 :: train_time 3392.1795259999926 complete
Run 8D :: seed 0 :: points 220 :: error 0.09799842205603362 :: train_time 3471.4249788000016 complete
Run 8D :: seed 0 :: points 240 :: error 0.11608119358711476 :: train_time 3069.1675557000563 complete
Run 8D :: seed 0 :: points 260 :: error 0.09454398973469515 :: train_time 1679.8442554999

KeyboardInterrupt: 

In [ ]:
ax = plot_error_measure(points[key],error[key],name = key)
#ax.set_ylim([0.01,1])


NameError: name 'points' is not defined

Complete a deep dive into the different dimensions for kriging

In [ ]:
for key, value in mfs.increasing_dimensions.items():
    my_function = value()
    X_test, y_test = create_test_points(my_function, n=2000)
    generate_model_comparisons(
        key,
        my_function,
        X_test=X_test,
        y_test=y_test,
        sample_low=10,
        sample_high=500,
        step_size=10,
        seeds=3,
        model_error=error,
        points=points,
    )

    

In [ ]:
# Plot the error for each erro
plot_error_measure(points["1D"],error["1D"],name = "1D",color="red",label="LBFGS")
plt.show()
plot_error_measure(points["2D"],error["2D"],name = "2D")
plt.show()
plot_error_measure(points["4D"],error["4D"],name = "4D")
plt.show()
plot_error_measure(points["6D"],error["6D"],name = "6D")
plt.show()
plot_error_measure(points["8D"],error["8D"],name = "8D")
plt.show()

In [ ]:
# Co-kriging comparison setup
from SurrogateModels import cokriging

def sample_hf_function(func, n, seed=0):
    # Create a set of samples to create a ground truth
    X_test = func.sample_random(n)
    y_test = func.high_fidelity(X_test)
    return X_test, y_test

def sample_lf_function(func, n, seed=0):
    # Create a set of samples to create a ground truth
    X_test = func.sample_random(n)
    y_test = func.low_fidelity(X_test)
    return X_test, y_test

def generate_mf_compare_cokriging(
    name,
    func: mfs.MultiFidelityTemplate,
    X_test,
    y_test,
    sample_low=5,
    sample_high=105,
    step_size=5,
    seeds=5,
    model_error = {},
    points = {}
):
    